<center>
<h1 style="font-family:verdana">
 💻 🧑 Reconeixement d'entitats anomenades 🧑 💻


<p> 🎯 <b>Objectiu</b>: en aquesta segona part de la pràctica aprendrem a reconèixer entitats anomenades, és a dir, identificar i classificar entitats en una oració que poden ser (en el context de reserva de vols) la ciutat de sortida o d'arribada, la data del mes i el dia, si es de negocis o turista o altres categories específiques. Aquesta tasca se sol plantejar etiquetant cada paraula amb una etiqueta de la categoria de l'entitat a la qual correspon.

Tant la classificació d'intencions (tasca estudiada a la primera part) com el reconeixement d'entitats anomenades (Name Entity Recognition, NER) que veurem en aquesta part són components crucials dels sistemes de processament del llenguatge natural (Natural Language Processing, NLP) i sovint s'utilitzen junts per crear aplicacions d'intel·ligència artificial (IA) conversacionals més sofisticades. Per exemple, en un xatbot, la classificació d'intencions ajuda a entendre la intenció principal de l'usuari, mentre que el reconeixement d'entitats anomenades ajuda a extreure entitats rellevants per proporcionar respostes més contextualitzades.


<p> ✨ <b>Contingut</b>: els passos d'aquesta segona part seran similars a la primera. En primer lloc, analitzarem el conjunt de dades. En segon lloc, prepararem les dades per a poder entrenar el model. I finalment dissenyarem l'arquitectura i entrenarem el model. </p>


<p> ✏ <b>Exercicis</b>: en cada secció anireu trobant exercicis que haureu d'anar resolent. </p>

---

<h2> Índex </h2>

1. [Inspecció del conjunt de dades](#section-one)
  * [Exercici 1](#ex-one)
  * [Exercici 2](#ex-two)
2. [Preprocessament de dades](#section-two)
  * [Exercici 3](#ex-three)
  * [Exercici 4](#ex-four)
3. [Disseny del model i entrenament](#section-three)
  * [Exercici 5](#ex-five)
  * [Exercici 6](#ex-six)
  * [Exercici 7](#ex-seven)
4. [Lliurable](#section-four)

---

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd "/content/drive/MyDrive/RUTA/A/TVD-repo/p2"
!ls

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import classification_report

<h1><a name="section-one"> 1. Inspecció del conjunt de dades </a></h1>

In [ ]:
!pip install gdown
!gdown "https://drive.google.com/uc?id=1u2wzXvsuscLeFHwXcDwMDaNDy0u_99-t"
!tar -zxf nlu_ATIS_data.tar.gz
!pip install seqeval

Downloading...
From: https://drive.google.com/uc?id=1u2wzXvsuscLeFHwXcDwMDaNDy0u_99-t
To: /content/nlu_ATIS_data.tar.gz
100% 122k/122k [00:00<00:00, 103MB/s]


In [16]:
!ls data

test.csv  train.csv


En aquesta segona part de la pràctica utilitzarem el mateix dataset que a la primera part, però en aquest cas utilitzarem la **primera columna** que correspon a les **oracions** introduïdes pels usuaris i la **segona columna** on trobarem les oracions d'entrada en format **BILOU**.

El format BILOU  és un esquema d'etiquetatge que es fa servir a les tasques de reconeixement d'entitats anomenades. El nom *BILOU* representa les etiquetes utilitzades en aquest esquema:

*   **B** - Beginning: primer token d'una entitat.
*   **I** - Inside: token dins d'una entitat.
*   **L** - Last: últim token de l'entitat.
*   **O** - Outside: token que no pertany a cap entitat.
*   **U** - Unit: entitats d'un sol token

A continuació, carregarem les dades per visualitzar i poder entendre millor aquestes etiquetes.

---

 <h1><a name="ex-one"><center> ✏ Exercici 1 ✏</a></h1>

A continuació us demanem que carregueu els dos CSVs de la carpeta `data`: `train.csv`, `test.csv` utilitzant pandas. Recorda que aquests CSVs no tenen capçalera. Agafeu les 900 últimes lineas del fitxer `train.csv` per crear un dataframe per validació.

In [17]:
# TODO
train_df = pd.read_csv('data/train.csv', header=None)
test_data = pd.read_csv('data/test.csv', header=None)
train_data = train_df.iloc[:-900]
val_data = train_df.iloc[-900:]

print('Training size:', len(train_data))
print('Validation dataset size:', len(val_data))
print('Test dataset size:', len(test_data))

Training size: 4078
Validation dataset size: 900
Test dataset size: 893


---

 <h1><a name="ex-two"><center> ✏ Exercici 2 ✏</a></h1>

Tal com hem introduït abans, per a aquest exercici ens centrarem en la **primera** i la **segona** columna. Per tant, ara us demanem que guardeu en les següents variables, les llistes corresponents a les oracions i a les etiquetes de les tres particions (`train`, `validation` i `test`).

In [18]:
# TODO
train_sentences = train_data[0].tolist()
train_labels = train_data[1].tolist()

val_sentences = val_data[0].tolist()
val_labels = val_data[1].tolist()

test_sentences = test_data[0].tolist()
test_labels = test_data[1].tolist()

---

Si tot ha anat bé ja podem analitzar quin aspecte té el format BILOU.

In [19]:
print(train_sentences[0])
print(train_labels[0])

i want to fly from boston at 838 am and arrive in denver at 1110 in the morning
 "O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day"


Aquest hauria de ser el resultat obtingut si executeu la cel·la anterior:

```
i want to fly from boston at 838 am and arrive in denver at 1110 in the morning
 "O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day"
```

En aquest exemple, `boston` i `838 am` corresponen a l'entitat "ciutat de sortida" i a l'entitat "hora de sortida" respectivament (`fromloc.city_name`,`depart_time.time`). `838` correspon al primer token que pertany la entitat `depart_time.time` i `am` al segon de la mateixa entitat. Els tokens com `i` o `want` no pertanyen a cap entitat.

In [20]:
print(train_sentences[1])
print(train_labels[1])

what flights are available from pittsburgh to baltimore on thursday morning
 "O O O O O B-fromloc.city_name O B-toloc.city_name O B-depart_date.day_name B-depart_time.period_of_day"


I aquest hauria de ser el resultat de la cel·la anterior:

```
what flights are available from pittsburgh to baltimore on thursday morning
 O O O O O B-fromloc.city_name O B-toloc.city_name O B-depart_date.day_name B-depart_time.period_of_day
```

En l'anterior exemple, `what`, `flights`, `are`, `available`, `from`, `to` i `on` estan etiquetades com a no pertanyents a cap entitat. I, en canvi, `pittsburgh` pertany a l'entitat d'un sol token anomenada ciutat d'arribada (`toloc.city_name`). També hi ha entitats compostes com `baltimore` que pertany a l'entitat ciutat de sortida (`fromloc.city_name`) i `thursday morning` que pertanyen a les entitats data de sortida (`depart_date.day_name`, `depart_time.period_of_day`).

Vegem-ne la llista completa d'entitats.

In [21]:
def count_unique_entities(list_of_label_sentences):
  flat_labels = []
  for labels in list_of_label_sentences:
    flat_labels += labels.split()
  c = Counter(flat_labels)
  return len(c), list(c.keys())

num_unique_entities, unique_entities = count_unique_entities(train_labels)

print("Number of different entities:", num_unique_entities)
print("Unique entities:", unique_entities)

Number of different entities: 216
Unique entities: ['"O', 'O', 'B-fromloc.city_name', 'B-depart_time.time', 'I-depart_time.time', 'B-toloc.city_name', 'B-arrive_time.time', 'B-arrive_time.period_of_day"', 'B-depart_date.day_name', 'B-depart_time.period_of_day"', 'B-flight_time', 'I-flight_time', 'I-fromloc.city_name', 'B-fromloc.city_name"', '"B-cost_relative', 'B-toloc.city_name"', '"B-round_trip', 'I-round_trip', 'B-cost_relative', 'B-fare_amount', 'I-fare_amount"', 'B-depart_date.today_relative', 'I-toloc.city_name', 'B-depart_date.day_name"', 'B-city_name"', 'I-toloc.city_name"', 'B-stoploc.city_name"', 'B-toloc.airport_code"', 'B-depart_time.time_relative', 'I-depart_time.time"', 'B-class_type', 'I-class_type', 'B-depart_date.date_relative', 'O"', 'B-airline_name', 'I-airline_name"', '"B-city_name', 'B-arrive_time.time_relative', 'B-arrive_time.time"', 'B-round_trip', 'I-airline_name', 'B-depart_time.start_time', 'I-depart_time.start_time', 'B-depart_time.end_time', 'I-depart_time

<h1><a name="section-two"> 2. Preprocessament de dades </a></h1>

El processament de les dades serà semblant al de la primera part d'aquesta pràctica.

---

 <h1><a name="ex-three"><center> ✏ Exercici 3 ✏</a></h1>

En aquest exercici us demanem que realitzeu els passos següents per preparar les dades.

 1. El primer pas serà construir el vocabulari a partir de les paraules presents a les oracions d'entrenament.

 2. El segon pas serà convertir les oracions en seqüències de nombres enters usant un diccionari o tokenitzador.

 3. El tercer pas serà guardar la longitud original de cada oració. Això ens serà útil per avaluar el nostre model sense tenir en compte el padding.

 4. Finalment, per aconseguir que totes les seqüències tinguen la mateixa longitud, fixarem la longitud segons la màxima trobada a l'entrenament i afegirem zeros a les oracions de menor longitud (padding).


In [22]:
# TODO
# Build vocabulary
word2idx = {'<pad>': 0, '<unk>': 1}
for sent in train_sentences:
    for word in sent.split():
        if word not in word2idx:
            word2idx[word] = len(word2idx)

max_sequence_length = max(len(sent.split()) for sent in train_sentences)

# Helper function to convert and pad sequences
def process_sentences(sentences, max_len):
    lengths = [len(s.split()) for s in sentences]
    padded_seqs = []
    for s in sentences:
        tokens = [word2idx.get(w, word2idx['<unk>']) for w in s.split()]
        padded = tokens + [word2idx['<pad>']] * (max_len - len(tokens))
        padded_seqs.append(padded[:max_len])
    return np.array(padded_seqs, dtype=np.int32), lengths

train_pad_sequences, len_train_sequences = process_sentences(train_sentences, max_sequence_length)
val_pad_sequences, len_val_sequences = process_sentences(val_sentences, max_sequence_length)
test_pad_sequences, len_test_sequences = process_sentences(test_sentences, max_sequence_length)

In [23]:
train_pad_sequences[0]

array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  8, 15, 13, 16,
       17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], dtype=int32)

Tal com vam fer a la primera part, aquí també hem de convertir les diferents classes d'entitats en índexs o vectors numèrics.

---

 <h1><a name="ex-four"><center> ✏ Exercici 4 ✏</a></h1>

Per aconseguir-ho haureu de seguir els passos següents.

 1. En primer lloc, haureu d'esbrinar quantes etiquetes diferents hi ha. Podeu prendre com a exemple la funció `count_unique_entities` per fer-ho. Tingueu en compte que les diferents etiquetes BILOU han de ser comptades com a classes individuals.

 2. El segon pas serà codificar les diferents classes trobades en etiquetes numèriques. Tingueu en compte que cada paraula de l'oració té una etiqueta i, per tant, per a cada oració tindrem una llista d'etiquetes. El *padding* es codificarà amb un índex dedicat (o amb l'etiqueta corresponent a **O**).

 3. Finalment, prepareu les estructures de dades o tensors PyTorch adequats per entrenar el model.


In [24]:
# TODO
# Build label mapping
label2idx = {'<pad>': 0}
for labels in train_labels:
    for lbl in labels.split():
        if lbl not in label2idx:
            label2idx[lbl] = len(label2idx)

idx2label = {i: l for l, i in label2idx.items()}

# Helper function to encode and pad labels
def encode_pad_labels(label_sentences, max_len):
    padded_labels = []
    for l_sent in label_sentences:
        tokens = [label2idx.get(l, label2idx['<pad>']) for l in l_sent.split()]
        padded = tokens + [label2idx['<pad>']] * (max_len - len(tokens))
        padded_labels.append(padded[:max_len])
    return np.array(padded_labels, dtype=np.int64)

train_pad_labels = encode_pad_labels(train_labels, max_sequence_length)
val_pad_labels = encode_pad_labels(val_labels, max_sequence_length)
test_pad_labels = encode_pad_labels(test_labels, max_sequence_length)

print("Number of different entities:", len(label2idx))
print("Unique entities:", list(label2idx.keys()))

Number of different entities: 217
Unique entities: ['<pad>', '"O', 'O', 'B-fromloc.city_name', 'B-depart_time.time', 'I-depart_time.time', 'B-toloc.city_name', 'B-arrive_time.time', 'B-arrive_time.period_of_day"', 'B-depart_date.day_name', 'B-depart_time.period_of_day"', 'B-flight_time', 'I-flight_time', 'I-fromloc.city_name', 'B-fromloc.city_name"', '"B-cost_relative', 'B-toloc.city_name"', '"B-round_trip', 'I-round_trip', 'B-cost_relative', 'B-fare_amount', 'I-fare_amount"', 'B-depart_date.today_relative', 'I-toloc.city_name', 'B-depart_date.day_name"', 'B-city_name"', 'I-toloc.city_name"', 'B-stoploc.city_name"', 'B-toloc.airport_code"', 'B-depart_time.time_relative', 'I-depart_time.time"', 'B-class_type', 'I-class_type', 'B-depart_date.date_relative', 'O"', 'B-airline_name', 'I-airline_name"', '"B-city_name', 'B-arrive_time.time_relative', 'B-arrive_time.time"', 'B-round_trip', 'I-airline_name', 'B-depart_time.start_time', 'I-depart_time.start_time', 'B-depart_time.end_time', 'I-de

<h1><a name="section-three"> 3. Disseny del model i entrenament </a></h1>

---

 <h1><a name="ex-five"><center> ✏ Exercici 5 ✏</a></h1>

En aquest exercici us demanem que dissenyeu una xarxa neuronal recurrent que prengui com a entrada les oracions codificades i retorni les etiquetes predites per a cada paraula. La xarxa ha de constar de les següents capes:

 1. Capa d'Embedding: per transformar els índexs de paraula en vectors densos.

 2. Capa Recurrent (LSTM o GRU, opcionalment bidireccional): per capturar el context de l'oració.

 3. Capa Densa lineal per a la classificació token per token.


In [25]:
class NERDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

class NERModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, bidirectional=True):
        super(NERModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=bidirectional)
        dirs = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * dirs, num_classes)

    def forward(self, x):
        embeds = self.embedding(x)
        lstm_out, _ = self.lstm(embeds)
        logits = self.fc(lstm_out)
        return logits

---

 <h1><a name="ex-six"><center> ✏ Exercici 6 ✏</a></h1>

Entreneu el model anterior, avalueu el rendiment del conjunt de validació/test i mostreu les mètriques principals de classificació token per token.

In [26]:
train_dataset = NERDataset(train_pad_sequences, train_pad_labels)
val_dataset = NERDataset(val_pad_sequences, val_pad_labels)
test_dataset = NERDataset(test_pad_sequences, test_pad_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NERModel(
    vocab_size=len(word2idx),
    embedding_dim=128,
    hidden_dim=128,
    num_classes=len(label2idx),
    bidirectional=True
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop with Validation Loss Computation
epochs = 10
for epoch in range(epochs):
    # --- Training Phase ---
    model.train()
    total_train_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs.view(-1, outputs.shape[-1]), batch_y.view(-1))
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- Validation Phase ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            val_loss = criterion(outputs.view(-1, outputs.shape[-1]), batch_y.view(-1))
            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# --- Test Evaluation ---
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        preds = torch.argmax(outputs, dim=-1).cpu().numpy()
        labels = batch_y.numpy()
        for p, l in zip(preds, labels):
            for p_i, l_i in zip(p, l):
                if l_i != 0:  # Ignore padding
                    y_true.append(l_i)
                    y_pred.append(p_i)

# Filter present labels and map names dynamically
present_labels = sorted(list(set(y_true) | set(y_pred)))
target_names = [idx2label[i] for i in present_labels]

print("\nTest Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=present_labels,
    target_names=target_names,
    zero_division=0
))


Epoch 01/10 | Train Loss: 1.7385 | Val Loss: 0.6753
Epoch 02/10 | Train Loss: 0.4466 | Val Loss: 0.3156
Epoch 03/10 | Train Loss: 0.2279 | Val Loss: 0.2021
Epoch 04/10 | Train Loss: 0.1399 | Val Loss: 0.1470
Epoch 05/10 | Train Loss: 0.0923 | Val Loss: 0.1159
Epoch 06/10 | Train Loss: 0.0639 | Val Loss: 0.0976
Epoch 07/10 | Train Loss: 0.0448 | Val Loss: 0.0857
Epoch 08/10 | Train Loss: 0.0328 | Val Loss: 0.0772
Epoch 09/10 | Train Loss: 0.0237 | Val Loss: 0.0718
Epoch 10/10 | Train Loss: 0.0175 | Val Loss: 0.0677

Test Classification Report:
                               precision    recall  f1-score   support

                           "O       0.99      1.00      0.99       841
                            O       0.99      1.00      0.99      4614
          B-fromloc.city_name       0.98      1.00      0.99       657
           B-depart_time.time       0.91      0.96      0.94        54
           I-depart_time.time       0.94      0.94      0.94        16
            B-toloc.city

---

 <h1><a name="ex-seven"><center> ✏ Exercici 7 ✏ </a></h1>

Modifiqueu els següents paràmetres del model anterior i analitzeu com afecten a la seva *accuracy*:

 1. **Mida dels Embeddings.** Proveu diferents mides d'*Embeddings* i observeu com canvia l'*accuracy* del model. Heu d'explicar les vostres conclusions.

 2. **Xarxes Convolucionals.** Afegiu capes convolucionals al vostre model. Expliqueu amb detall els valors que heu provat i la vostra motivació a l'hora d'escollir-los. Recordeu, que també podeu provar diferents configuracions de *pooling*.

 3. **Xarxes Recurrents.**  Afegiu capes recurrents al vostre model (LSTM, GRU). Expliqueu amb detall els valors que heu provat i la vostra motivació.

 4. ** Transformer.** Afegiu blocs de Transformer al vostre model. Expliqueu amb detall els valors que heu provat i la vostra motivació.

 5. **Regularització.** Quan proveu configuracions amb més paràmetres veureu que el model comença a tenir *overfitting* molt prompte durant l'entrenament. Afegiu *Dropout* al vostre model. Heu d'explicar la vostra decisió de valors i de posició dins de la xarxa.


 6. **Balancejat de les classes.** Si analitzeu el dataset, veureu que la freqüència de les classes està molt desbalancejada. Podeu consultar la documentació de la loss per veure com fer-ho (https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html)

 ---

---

## Exercici 7: metodologia

Tot el codi de l'Exercici 7 és al paquet `parte2/ner/`: configuracions (`config.py`), dades (`data.py`), models
compostos per blocs (`models/`), un únic bucle d'entrenament (`train.py`), mètriques (`metrics.py`) i una
graella per apartat (`experiments/`). Totes les variants comparteixen preprocessament, entrenament i avaluació,
així que les diferències entre resultats només poden venir del factor que es canvia.

El protocol és el mateix a tots els apartats. Es canvia un factor cada vegada respecte al baseline de
l'Exercici 6, cada configuració s'entrena amb 3 llavors i es reporta la mitjana ± la desviació, i totes les
decisions es prenen amb la F1 de validació (el test només s'observa). La mètrica principal és la F1 d'entitats
en mode estricte (seqeval, equivalent a conlleval); també es mostren la F1 macro, que reflecteix les classes
rares, el nombre de paràmetres i el temps per època. El baseline es reentrena amb early stopping més robust
(`min_delta = 0.001` i fins a 40 èpoques), perquè a l'Exercici 6 la F1 encara pujava mil·lèsimes a l'època 30.

Cada execució es desa en un JSON a `parte2/results/` en acabar, de manera que si l'entorn es desconnecta, en
tornar a executar la cel·la només s'entrena el que falta.

In [ ]:
# Configuració de l'Exercici 7. Els resultats es desen a parte2/results; si p2/ és a Google Drive, sobreviuen
# a les desconnexions de Colab.
import os
import sys
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    display = print

# El notebook s'executa des de la carpeta p2/ (a Colab, després de muntar Drive i fer %cd a p2/).
PARTE2_DIR = Path("parte2").resolve()
assert (PARTE2_DIR / "ner").exists(), f"No trobo el paquet ner a {PARTE2_DIR}: el directori actual ha de ser p2/"

os.environ["NER_RESULTS_DIR"] = str(PARTE2_DIR / "results")
sys.path.insert(0, str(PARTE2_DIR))

from ner import run_grid
from ner.experiments import ex7_1, ex7_2, ex7_3, ex7_4, ex7_5, ex7_6, extras, final

# QUICK = True entrena 1 època amb 1 llavor: només serveix per comprovar que tot s'executa.
# El codi de ner/ s'ha de congelar abans de llançar les graelles de debò: la cache identifica cada execució
# per la seva configuració, no per la versió del codi.
QUICK = False
N_WORKERS = 2            # execucions simultànies a la GPU; es pot ajustar amb el benchmark de la cel·la següent
SEEDS = (0,) if QUICK else (0, 1, 2)
FINAL_SEEDS = (0,) if QUICK else final.FINAL_SEEDS


def run(configs, seeds=SEEDS):
    """Llança una graella amb cache; en mode QUICK redueix cada configuració a una sola època."""
    if QUICK:
        configs = [c.with_(**{"train.max_epochs": 1}) for c in configs]
    return run_grid(configs, seeds=seeds, n_workers=N_WORKERS)


def fig_name(name):
    """En mode QUICK no es desen figures, per no sobreescriure les bones amb resultats d'una època."""
    return None if QUICK else name

In [ ]:
# Benchmark opcional: quantes execucions simultànies aprofiten millor la GPU. Entrena 4 models curts per a cada
# valor en una carpeta temporal, sense tocar la cache de resultats.
import tempfile
import time

from ner import baseline

if not QUICK:
    results_dir = os.environ["NER_RESULTS_DIR"]
    for workers in (1, 2, 4):
        with tempfile.TemporaryDirectory() as tmp:
            os.environ["NER_RESULTS_DIR"] = tmp
            start = time.perf_counter()
            run_grid([baseline().with_(**{"train.max_epochs": 3})], seeds=(10, 11, 12, 13), n_workers=workers)
            print(f"n_workers = {workers}: {time.perf_counter() - start:.0f} s")
    os.environ["NER_RESULTS_DIR"] = results_dir

### 7.1 Mida dels embeddings i embeddings preentrenats

Es proven embeddings aleatoris de 16 a 512 dimensions i GloVe (50, 100, 200 i 300) congelat i amb fine-tuning.
Hipòtesi: amb un vocabulari d'unes 600 paraules la F1 hauria de saturar cap a 50-100 dimensions i empitjorar
lleugerament per sobre de 256 per sobreajust. GloVe hauria d'ajudar sobretot amb paraules rares o no vistes
(Jurafsky & Martin, cap. 6). Com que el vocabulari amb GloVe també inclou les paraules que surten un sol cop a
train, el control `aleatori-100 min_freq=1` separa l'efecte del vocabulari del dels vectors preentrenats.

In [ ]:
display(ex7_1.vocab_stats().round(4))
r71 = run(ex7_1.configs())
s71, fig71 = ex7_1.report(r71, save_as=fig_name("ex7_1_embeddings"))
display(s71.round(4))

**Anàlisi dels resultats:** *pendent d'executar les graelles a Colab.*

### 7.2 Xarxes convolucionals i pooling

Tres usos de la convolució: una CNN de paraula davant de la BiLSTM, una CNN sola i una char-CNN amb max-pooling
sobre els caràcters de cada paraula (Ma & Hovy, 2016). Un global pooling sol no serveix en NER perquè elimina la
sortida per token; aquí el pooling es fa servir com a max-pooling local amb stride 1, com a vector de context
global concatenat a cada token i dins de la char-CNN. Hipòtesi: la CNN davant de la BiLSTM aporta poc perquè la
BiLSTM ja captura el context local; la CNN sola falla en etiquetes que depenen de paraules llunyanes
(`arrive_time` vs `depart_time`); la char-CNN millora les paraules fora de vocabulari.

In [ ]:
r72 = run(ex7_2.configs())
s72, fig72 = ex7_2.report(r72, save_as=fig_name("ex7_2_cnn"))
display(s72.round(4))

**Anàlisi dels resultats:** *pendent d'executar les graelles a Colab.*

### 7.3 Xarxes recurrents

Primer es canvia un factor cada vegada (tipus de cel·la, direcció, capes i mida de l'estat) i després es creuen
automàticament els guanyadors de cada factor amb el valor del baseline. La LSTM unidireccional de 256 unitats té
la mateixa mida de sortida que la BiLSTM de 128 i separa l'efecte de la direcció del de la capacitat. Hipòtesi:
LSTM i GRU queden a prop (oracions curtes), la bidireccionalitat és el factor més important i més capes sense
dropout sobreajusten.

In [ ]:
r73 = run(ex7_3.configs())
r73 += run(ex7_3.cross_configs(r73))
s73, fig73 = ex7_3.report(r73, save_as=fig_name("ex7_3_rnn"))
display(s73.round(4))

**Anàlisi dels resultats:** *pendent d'executar les graelles a Colab.*

### 7.4 Transformer

Un encoder Transformer sol (amb posició sinusoidal, apresa o sense posició) i un híbrid amb capes d'atenció
sobre la BiLSTM. La primera fase compara la recepta d'optimització del baseline amb LR 5e-4 i warmup; la segona
fa servir la recepta guanyadora a totes les variants. Hipòtesi: entrenat des de zero amb 4k oracions, el
Transformer sol queda per sota de la BiLSTM perquè no té biaix de seqüencialitat; sense posició falla en
distincions d'ordre com `fromloc`/`toloc`; l'híbrid iguala o millora lleugerament el baseline.

In [ ]:
r74 = run(ex7_4.configs())
r74 += run(ex7_4.variant_configs(r74))
s74, fig74 = ex7_4.report(r74, save_as=fig_name("ex7_4_transformer"))
display(s74.round(4))

**Anàlisi dels resultats:** *pendent d'executar les graelles a Colab.*

### 7.5 Regularització amb dropout

Dropout sobre l'embedding, entre capes recurrents, abans del classificador i dins del Transformer, en tres
models: el baseline, una BiLSTM 2×256 i el millor model amb atenció de l'apartat anterior. Cada posició es prova
sola (p = 0.3) i després totes juntes (p = 0.1, 0.3, 0.5). Hipòtesi: l'efecte més clar és al model gran (menys
distància entre la loss de train i la de validació i millor F1); p = 0.5 és massa per a un dataset tan petit.

In [ ]:
r75 = run(ex7_5.configs(r74))
s75, fig75, curves75 = ex7_5.report(r75, save_as=fig_name("ex7_5_dropout"))
display(s75.round(4))

**Anàlisi dels resultats:** *pendent d'executar les graelles a Colab.*

### 7.6 Balanceig de les classes

Pesos a la cross-entropy segons la freqüència de cada etiqueta a train: inversa (amb i sense límit), inversa de
l'arrel, nombre efectiu de mostres (Cui et al., 2019) i reduir només el pes de `O`. Hipòtesi: els pesos pugen el
recall i la F1 de les classes rares a canvi de precisió; la inversa pura és massa agressiva i el límit o l'arrel
donen el millor compromís.

In [ ]:
display(ex7_6.weights_table().round(3))
r76 = run(ex7_6.configs())
s76, fig76 = ex7_6.report(r76, save_as=fig_name("ex7_6_balanceig"))
display(s76.round(4))

**Anàlisi dels resultats:** *pendent d'executar les graelles a Colab.*

### Experiments addicionals: CRF, esquema d'etiquetes i vocabulari

Disseny 2×2 entre esquema (BIO, BILOU) i decodificador (softmax, CRF implementat a mà amb forward i Viterbi), i
`min_freq = 1`. Hipòtesi: amb softmax, BILOU produeix més seqüències impossibles que BIO, i el CRF, que aprèn
quines transicions són vàlides, les redueix gairebé a zero.

In [ ]:
rx = run(extras.configs())
sx, figx = extras.report(rx, save_as=fig_name("extra_crf_esquema"))
display(sx.round(4))

**Anàlisi dels resultats:** *pendent d'executar les graelles a Colab.*

### Model final, ablació i significança

El model final es construeix automàticament amb els resultats de validació: de cada apartat s'agafa la
configuració guanyadora i només s'hi afegeix si millora la seva referència per més d'una desviació estàndard.
Després, l'ablació treu un component cada vegada (5 llavors) i un bootstrap pareat sobre les oracions de test
diu si la millora respecte al baseline és significativa.

In [ ]:
section_results = {"7.1": r71, "7.2": r72, "7.3": r73, "7.4": r74, "7.5": r75, "7.6": r76, "extra": rx}
components, log = final.build_final(section_results)
print("\n".join(log))
final_cfg = final.compose(components)
print("\nModel final:", {k: v for k, v in final_cfg.to_dict().items() if k in ("data", "model")})
print("Entrenament:", final_cfg.to_dict()["train"])

rf = run(final.ablation_configs(components), seeds=FINAL_SEEDS)
sf, figf = final.report(rf, save_as=fig_name("final_ablacio"))
display(sf.round(4))
display(final.significance(rf).round(4))

In [ ]:
s_all, fig_all = final.overview_figure(section_results, rf, save_as=fig_name("resum_global"))
display(s_all[["test_entity_f1_mean", "test_entity_f1_std", "test_entity_f1_macro_mean", "n_params"]].round(4))

**Anàlisi dels resultats:** *pendent d'executar les graelles a Colab.*

### Anàlisi d'errors

Es fa servir l'execució amb F1 de validació mediana de cada model, per no triar la llavor més afortunada. Cada
error d'entitat es classifica com a tipus equivocat, límits equivocats, no detectada o inventada.

In [ ]:
from ner import analysis
from ner.config import ExperimentConfig
from ner.data import prepare_data
from ner.metrics import entity_report

categories = {}
for name in (final.BASELINE, final.FINAL):
    run_result = analysis.median_seed_result(rf, name)
    cfg = ExperimentConfig.from_dict(run_result["config"])
    test = prepare_data(cfg.data).splits["test"]
    pred = [p.split() for p in run_result["test_pred"]]
    cat, conf = analysis.error_summary(test.gold, pred)
    categories[name] = cat
    print(f"\n=== {name}: confusions de tipus més freqüents ===")
    display(conf)
display(pd.DataFrame(categories).round(3))

report_final = entity_report(test.gold, pred, cfg.data.scheme)
print("Entitats amb pitjor F1 al model final (amb suport mínim de 5):")
display(report_final[report_final["support"] >= 5].head(10).round(3))
for example in analysis.error_examples(test.sentences, test.gold, pred, n=4):
    print(example, end="\n\n")

**Anàlisi dels resultats:** *pendent d'executar les graelles a Colab.*

---

<h1><a name="section-four"> 4. Lliurable </a></h1>

Heu d'entregar un document PDF de com a **màxim 10 pàgines** que incloga els resultats de tots els exercicis així com una explicació de cadascun dels resultats i de la modificació que heu fet. L'estructura del document és:

1. Introducció.
2. Experiments i Resultats (amb raonament).
3. Conclusions.

No cal que afegiu el vostre codi al document, podeu entregar el *notebook* juntament amb el document.

 ---